# Soilgrid Extraction Notebook

Note on SoilGrid Data Extraction:
The SoilGrid API was used to fetch the public soil data for the unique coordinates. Because the Snowflake environment blocks external HTTP requests (NameResolutionError due to security policies), this specific extraction block was executed locally. The resulting data was saved as soil_features.csv and uploaded to the Snowflake environment to be merged with the main dataset. The original extraction code is preserved below for transparency and reproducibility.

## Environment preparation

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df

In [ ]:
# If you want to reproduce this code, set this variable to 'True'
RUN_API = False

In [ ]:
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
    [['Latitude', 'Longitude']]
)

validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
    [['Latitude', 'Longitude']]
)

unique_coords_df = pd.concat(
    [water_quality_df, validation_df],
    ignore_index=True
).drop_duplicates()

In [ ]:
soil_names = []
soil_purities = []
MAX_RETRIES = 3

if RUN_API:
    # Reset index of unique_coords_df to ensure iteration works correctly
    unique_coords_df = unique_coords_df.reset_index(drop=True)
    
    # Sending batch requests
    for i in range(0, len(unique_coords_df)):
        lat = unique_coords_df.loc[i, 'Latitude']
        lon = unique_coords_df.loc[i, 'Longitude']
    
        # Calls API
        url = f'https://rest.isric.org/soilgrids/v2.0/classification/query?lat={lat}&lon={lon}'
        
        # Retry control
        success = False
        
        for retry in range(MAX_RETRIES):
            try:
                response = requests.get(url)
                if response.status_code == 200:
                    data = response.json()
        
                    # Retrieve name
                    soil_names.append(data.get('wrb_class_name', 'Unknown'))
            
                    # Tries to retrieve probability
                    try:
                        purity = data['wrb_class_probability'][0][1]
                        soil_purities.append(purity)
                    except (IndexError, KeyError, TypeError):
                        soil_purities.append(np.nan)
            
                    success = True
                    break
                else:
                    time_sleep = 2 ** retry # Waits more fore each retry
                    print(f'⚠️ Error {response.status_code} on line {i}. Try {retry + 1}/{MAX_RETRIES}. Trying again on {time_sleep}s...')
                    time.sleep(time_sleep)
        
            except Exception as e:
                time_sleep = 2 ** retry # Waits more fore each retry
                print(f'⚠️ Error {response.status_code} on line {i}. Try {retry + 1}/{MAX_RETRIES}. Trying again on {time_sleep}s...')
                time.sleep(time_sleep)
        
        if not success:
            print(f'❌ Complete fail on line {i} after {MAX_RETRIES} tries. Filling up with Nan.')
            soil_names.append('Unknown')
            soil_purities.append(np.nan)
        
        # Waits for another request
        time.sleep(0.5)
    
    # Adds new column
    unique_coords_df['soil_name'] = soil_names
    unique_coords_df['soil_purity'] = soil_purities
    
    # Saves dataframe
    save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/intermediate/'
    save_df(unique_coords_df, 'elevation_feature.csv', save_path)